# Information Extraction: The Data Miner

**Project Brief**

Welcome, **Data Miner**.

We are buried under mountains of unstructured text (ore). Your job is to extract valuable, structured data: **Entities, Relations, and Events**.

Your Job:
1.  **Relation Extraction**: Find how entities connect (The Vein).
2.  **Event Extraction**: Identify who did what to whom (The Nuggets).
3.  **Temporal Analysis**: Carbon-date the events (Time).
4.  **Template Filling**: Smelt the ore into structured records.

---

In [1]:
# Mining Tools (Setup)
!pip install spacy nltk pandas numpy
!python -m spacy download en_core_web_sm

import spacy
from spacy import displacy
import pandas as pd
import re
import nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 11.4 MB/s  0:00:01m0:00:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


## 1. Relation Extraction (The Vein)

We want to find tuples like `(Bill Gates, FOUNDER_OF, Microsoft)`.

**Hearst Patterns** (Rule-Based Mining):
Specific phrasing often indicates a relationship. 
*   "X, **such as** Y" -> `(Y, IS_A, X)`
*   "X, **including** Y" -> `(Y, IS_A, X)`

### Precision vs. Recall Tradeoff
Mining is a balance:
*   **High Precision**: Your rules are strict (e.g., regex). You find pure gold, but you miss a lot of gold dust (Low Recall).
*   **High Recall**: Your rules are loose (e.g., "X and Y appear in same sentence"). You find all the gold, but tons of dirt too (Low Precision).

In [ ]:
# Implementing Hearst Patterns

text = "Tech giants such as Apple and Microsoft are investing heavily. Other companies, including Tesla, are also relevant."

def extract_is_a(text):
    relations = []
    
    # Pattern: X such as Y
    # Note: This is a simplified regex. Real extractions needs full parsing.
    # (\w+) = Capture word (X)
    # such as 
    # (\w+) = Capture word (Y)
    pattern1 = re.compile(r'(\w+)\W+such as\W+(\w+)')
    for match in pattern1.finditer(text):
        relations.append((match.group(2), 'IS_A', match.group(1)))
        
    # Pattern: X, including Y
    pattern2 = re.compile(r'(\w+)\W+including\W+(\w+)')
    for match in pattern2.finditer(text):
        relations.append((match.group(2), 'IS_A', match.group(1)))
        
    return relations

print("Extracted Relations:")
for r in extract_is_a(text):
    print(f"{r[0]} -> {r[1]} -> {r[2]}")

## 2. Temporal Analysis (Carbon Dating)

Events happen in time. We need to normalize time expressions.
*   "Next Tuesday"
*   "Three years ago"

ISO 8601 Standard: `YYYY-MM-DD`

To do this robustly, we usually need libraries (like `dateparser` or `sutime`), but let's build a simple normalizer.

In [ ]:
import datetime

def normalize_date(text_date, relative_to=None):
    if relative_to is None: relative_to = datetime.date.today()
    
    text_date = text_date.lower()
    
    if "yesterday" in text_date:
        return relative_to - datetime.timedelta(days=1)
    elif "tomorrow" in text_date:
        return relative_to + datetime.timedelta(days=1)
    # Simple case: "X days ago"
    elif "days ago" in text_date:
        days = int(text_date.split()[0])
        return relative_to - datetime.timedelta(days=days)
        
    return "Unresolved"

print(f"Today: {datetime.date.today()}")
print(f"'Yesterday' -> {normalize_date('Yesterday')}")
print(f"'5 days ago' -> {normalize_date('5 days ago')}")

## 3. Template Filling (The Smelter)

The goal is to extract structured records from financial texts.

**Scenario**: Merger & Acquisition (M&A) extraction.
**Fields**: `Buyer`, `Target`, `Amount`.

In [ ]:
# The Ore
news_snippets = [
    "Google acquired YouTube for $1.65 billion in 2006.",
    "In 2012, Facebook bought Instagram for $1 billion.",
    "Microsoft purchases LinkedIn for $26.2 billion."
]

def smelt_ore(texts):
    records = []
    # Regex heuristics (Very brittle, but demonstrates concept)
    # X acquired Y for Z
    # X bought Y for Z
    # X purchases Y for Z
    
    pattern = re.compile(r'(.*?)\s+(?:acquired|bought|purchases)\s+(.*?)\s+for\s+(\$[\d\.]+\s+\w+)')
    
    for text in texts:
        match = pattern.search(text)
        if match:
            records.append({
                'Buyer': match.group(1),
                'Target': match.group(2),
                'Amount': match.group(3)
            })
            
    return pd.DataFrame(records)

df = smelt_ore(news_snippets)
print(df)

## 4. Coreference Resolution (Who is 'He'?)

To extract deep meaning, we must resolve pronouns.
"**Bill Gates** founded Microsoft. **He** is rich."

If we don't know `He = Bill Gates`, we miss the fact `(Bill Gates, IS, rich)`.
Coreference Resolution groups mentions (`Bill Gates`, `He`, `The founder`) into a single Entity Cluster.

## 5. Evaluation
Evaluating IE systems is generally done by Precision, Recall, and F1 at the tuple level.
Did we extract `(Google, YouTube)` correctly?

**Project Completed, Data Miner.** The ore has been processed into gold bars.